Function to create prompt,target

In [1]:
def create_dataset_with_prompt_target(example):
  observations=example['observations']
  obs=''
  possibilities=example['possible_scenario']
  poss=''
  impossibilities=example['impossible_scenario']
  imposs=''
  for i,impossibility in enumerate(impossibilities):
    imposs+=f"{i+1} {impossibility['scenario'].strip('.')} because {impossibility['reason']}\n"
  for i,possibility in enumerate(possibilities):
    poss+=f"{i+1} {possibility}\n"
  for i,observation in enumerate(observations):
    obs+=f"{i+1} {observation}\n"
  prompt="<<STATEMENT>>\n"+example['problem']
  target=f"<<OBSERVATIONS>>\n{obs}<<POSSIBILITIES>>\n{poss}<<IMPOSSIBILITIES>>\n{imposs}<<DEDUCTION>>\n{example['deduction']}\n<<FINAL ANSWER>>\n{example['final_answer']}"
  return {'prompt':prompt,'target':target,'context':example['context'],'difficulty':example['difficulty']}


In [2]:
example={"problem": "At 8 PM, the owner of the downtown bakery found the cash register empty. The security‑camera footage shows the delivery man, Mr. Patel, entering the bakery at 7:45 PM and leaving at 7:50 PM, carrying a large box. The bakery’s alarm system was not triggered.", "observations": ["The cash register was empty at 8 PM.", "Security footage shows Mr. Patel entering at 7:45 PM.", "Security footage shows Mr. Patel leaving at 7:50 PM carrying a large box.", "The alarm system was not triggered."], "possible_scenario": ["The delivery man, Mr. Patel, stole the cash from the register.", "The bakery owner misplaced the cash after closing.", "A customer took the cash during the closing shift."], "impossible_scenario": [{"scenario": "The bakery owner misplaced the cash after closing", "reason": "The only person recorded entering after business hours was Mr. Patel; no evidence of the owner handling the register after closing."}, {"scenario": "A customer took the cash during the closing shift", "reason": "No customer was seen on camera after the store closed; only Mr. Patel entered and left before 8 PM."}], "deduction": "The observations show that the cash register was empty when the owner checked at 8 PM. The only individual captured on the security camera entering the bakery after normal hours was the delivery man, Mr. Patel, who was present from 7:45 PM to 7:50 PM carrying a large box, and the alarm never went off, indicating no forced entry. Because no other person was recorded inside the premises during that window, the most direct explanation for the missing cash is that Mr. Patel took it while he was inside.", "final_answer": "The cash was stolen by the delivery man, Mr. Patel.", "context": "crime/mystery scenario", "difficulty": 1}
create_dataset_with_prompt_target(example)

{'prompt': '<<STATEMENT>>\nAt 8\u202fPM, the owner of the downtown bakery found the cash register empty. The security‑camera footage shows the delivery man, Mr. Patel, entering the bakery at 7:45\u202fPM and leaving at 7:50\u202fPM, carrying a large box. The bakery’s alarm system was not triggered.',
 'target': '<<OBSERVATIONS>>\n1 The cash register was empty at 8\u202fPM.\n2 Security footage shows Mr. Patel entering at 7:45\u202fPM.\n3 Security footage shows Mr. Patel leaving at 7:50\u202fPM carrying a large box.\n4 The alarm system was not triggered.\n<<POSSIBILITIES>>\n1 The delivery man, Mr. Patel, stole the cash from the register.\n2 The bakery owner misplaced the cash after closing.\n3 A customer took the cash during the closing shift.\n<<IMPOSSIBILITIES>>\n1 The bakery owner misplaced the cash after closing because The only person recorded entering after business hours was Mr. Patel; no evidence of the owner handling the register after closing.\n2 A customer took the cash during

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Writing prompt,target in json file

In [4]:
import json
new_file='/content/drive/MyDrive/dataset_for_SMOLLM/dataset_in_prompt_target_format.jsonl'
dataset='/content/drive/MyDrive/dataset_for_SMOLLM/dataset_filtered.jsonl'
with open(dataset, "r", encoding="utf-8") as f, open(new_file, "w", encoding="utf-8") as w:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            example = json.loads(line)

        except json.JSONDecodeError as e:
            print(f"Line {i} failed to parse: {e}")
            continue

        edited = create_dataset_with_prompt_target(example)
        json_line=json.dumps(edited,ensure_ascii=False)
        w.write(json_line+'\n')

In [6]:
tokenizer(example['prompt'], add_special_tokens=False)==tokenizer(example['prompt'], add_special_tokens=True)

True

In [10]:
def build_labels(example):
  prompt_ids = tokenizer(example['prompt'], add_special_tokens=False)["input_ids"]
  #print(len(prompt_ids))
  target_ids = tokenizer(example['target'], add_special_tokens=False)["input_ids"]
  input_ids = prompt_ids + target_ids + [tokenizer.eos_token_id]
  labels=len(prompt_ids)*[-100]+target_ids + [tokenizer.eos_token_id]
  return {'input_ids':input_ids,'labels':labels,'context':example['context'],'difficulty':example['difficulty']}

output=build_labels({"prompt": "<<STATEMENT>>\nAt 9:00 AM the bank vault safe was found open. The security camera footage shows only the night guard, who was on his scheduled break at that time. The guard’s keycard was recorded as the one used to unlock the vault.", "target": "<<OBSERVATIONS>>\n1 The vault safe was open at 9:00 AM\n2 The security camera captured only the night guard during the break period\n3 The guard’s keycard was used to unlock the vault\n<<POSSIBILITIES>>\n1 The night guard opened the safe himself during his break\n2 An accomplice used the guard’s keycard to open the safe\n<<IMPOSSIBILITIES>>\n1 An accomplice used the guard’s keycard to open the safe because No observation of any other person in the camera footage; only the guard was seen\n<<DEDUCTION>>\nFrom the observations we know the safe was open at 9:00 AM, the guard’s keycard unlocked it, and the camera shows no one besides the guard while he was on break. Therefore any theory that requires another person (an accomplice) fails because the footage provides no evidence of anyone else being present. The only scenario consistent with all observed facts is that the guard himself used his keycard to open the vault during his break.\n<<FINAL ANSWER>>\nThe night guard opened the safe himself during his break.", "context": "crime/mystery scenario", "difficulty": 1})
print(output['input_ids'])
print(output['labels'])
print(output['context'])
print(output['difficulty'])

[33691, 23438, 26040, 7791, 198, 3473, 216, 41, 42, 32, 32, 321, 124, 4153, 260, 5461, 24439, 2991, 436, 983, 1440, 30, 378, 3292, 6715, 22464, 2744, 805, 260, 3163, 8246, 28, 617, 436, 335, 650, 13831, 2373, 418, 338, 655, 30, 378, 8246, 417, 99, 1646, 12172, 436, 5797, 347, 260, 582, 804, 288, 16271, 260, 24439, 30, 33691, 63, 7770, 22840, 28074, 7791, 198, 33, 378, 24439, 2991, 436, 1440, 418, 216, 41, 42, 32, 32, 321, 124, 4153, 198, 34, 378, 3292, 6715, 8025, 805, 260, 3163, 8246, 981, 260, 2373, 1657, 198, 35, 378, 8246, 417, 99, 1646, 12172, 436, 804, 288, 16271, 260, 24439, 198, 33691, 22259, 9114, 50, 4192, 45787, 7791, 198, 33, 378, 3163, 8246, 6572, 260, 2991, 3930, 981, 650, 2373, 198, 34, 1117, 5736, 569, 804, 260, 8246, 417, 99, 1646, 12172, 288, 1440, 260, 2991, 198, 33691, 5974, 22259, 9114, 50, 4192, 45787, 7791, 198, 33, 1117, 5736, 569, 804, 260, 8246, 417, 99, 1646, 12172, 288, 1440, 260, 2991, 975, 2838, 7928, 282, 750, 550, 1055, 281, 260, 6715, 22464, 43, 805, 26

In [18]:
len(output['input_ids'])==len(output['labels'])

True

Loading dataset and spliting

In [13]:
from datasets import load_dataset
dataset=load_dataset("json",data_files='/content/drive/MyDrive/dataset_for_SMOLLM/dataset_in_prompt_target_format.jsonl')['train']
dataset=dataset.map(build_labels)
split=dataset.train_test_split(test_size=120,seed=1)
dataset_train=split['train']
dataset_val=split['test']


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/976 [00:00<?, ? examples/s]

In [14]:
print(len(dataset_train))
print(len(dataset_val))

856
120


In [17]:
print(dataset_train[0])

{'prompt': '<<STATEMENT>>\nA sealed glass beaker containing a clear liquid is placed on a hot plate set to 100°C. After 5 minutes, the liquid has boiled and the beaker is now empty, but no steam is observed rising from the beaker.', 'target': '<<OBSERVATIONS>>\n1 The beaker is sealed.\n2 The liquid inside the beaker is clear.\n3 The hot plate temperature is set to 100°C.\n4 After 5 minutes the liquid has boiled.\n5 The beaker is now empty.\n6 No steam is observed rising from the beaker.\n<<POSSIBILITIES>>\n1 The liquid was water, and its vapor condensed inside the sealed beaker.\n2 The liquid was a low‑boiling alcohol that escaped through a tiny leak in the seal.\n3 The liquid was mercury, which boiled at the hot plate temperature.\n<<IMPOSSIBILITIES>>\n1 The liquid was a low‑boiling alcohol that escaped through a tiny leak in the seal because The beaker is sealed, and no steam was observed; a leak would have allowed vapor to escape and be visible.\n2 The liquid was mercury, which boil